# Phase 10 - BERT Embedding Sidebar (Novelty)

Goal: encode complaint descriptions with a pretrained sentence-transformer, run K-Means on those embeddings, and compare cluster quality to Phase 5's Word2Vec result. This is the project's modernization story — Phase 5 saw Word2Vec quality limited by our 1,187-word vocabulary; pretrained transformer embeddings should give us much richer semantics with zero additional training.

Model: `sentence-transformers/all-MiniLM-L6-v2` — 80MB, 384-dim outputs, ~14M parameters. Encodes ~1000 short docs/sec on a Colab T4 GPU.

**Switch the runtime to T4 GPU** before running this notebook (Runtime -> Change runtime type -> GPU). On CPU this phase takes 20x longer.

Phase 2's `sample_2m_preprocessed.parquet` must be on Drive. Phase 5 must have completed (we compare against its silhouette score).

## Cell 1 - Bootstrap (with GPU verification)

In [ ]:
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

!pip install -r /content/project/requirements.txt -q

# verify gpu - if cuda is unavailable, encoding will fall back to cpu and take ~20x longer
import torch
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
    print(f'gpu memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: no gpu detected. switch runtime to GPU or this will take ~30 min instead of ~3.')

## Cell 2 - Load preprocessed data + sample

We encode 100K rows. The full 1.94M would take 30+ min and the resulting parquet would be 600+ MB which exceeds Streamlit Cloud's free tier limit. 100K is plenty for clustering quality and stays under 300 MB.

In [ ]:
# we DO NOT need spark for the encoding step - pandas is enough.
# saves us the spark startup cost.
import pandas as pd
from pathlib import Path

in_path = '/content/drive/MyDrive/cs6513/sample_2m_preprocessed.parquet'

# read parquet directly with pyarrow - faster than spark for this size
df = pd.read_parquet(in_path, columns=['unique_key', 'label_canonical', 'problem_detail', 'tokens'])
df = df[df['problem_detail'].notna() & (df['problem_detail'].str.len() >= 3)]
print(f'loaded {len(df):,} rows from preprocessed parquet')

# stratified sample 100K rows so all top categories are represented
from src.config import TOP_K_CATEGORIES
top_classes = df['label_canonical'].value_counts().head(TOP_K_CATEGORIES).index.tolist()
df = df[df['label_canonical'].isin(top_classes)]

TARGET = 100_000
frac = TARGET / len(df)
df_sample = df.groupby('label_canonical', group_keys=False).apply(
    lambda g: g.sample(frac=min(1.0, frac), random_state=42)
).reset_index(drop=True)
print(f'stratified sample size: {len(df_sample):,}')

## Cell 3 - Load sentence-transformers + encode

First call downloads the 80MB model from huggingface. After that its instant. We encode in batches of 256 for GPU throughput.

In [ ]:
from sentence_transformers import SentenceTransformer
import time, numpy as np

model_name = 'sentence-transformers/all-MiniLM-L6-v2'
encoder = SentenceTransformer(model_name)
if torch.cuda.is_available():
    encoder = encoder.to('cuda')

texts = df_sample['problem_detail'].astype(str).tolist()

t0 = time.time()
embeddings = encoder.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
t_encode = time.time() - t0
print(f'\nencoded {len(texts):,} docs in {t_encode:.1f} sec ({len(texts)/t_encode:.0f} docs/sec)')
print(f'embedding shape: {embeddings.shape}')
print(f'embedding size in memory: {embeddings.nbytes / 1024 / 1024:.1f} MB')

## Cell 4 - Probe semantic similarity

Pretrained transformers are supposed to capture richer semantics than co-occurrence-trained Word2Vec. We test by finding the nearest descriptions to a probe phrase.

In [ ]:
probe_phrases = [
    'rat infestation in the building',
    'loud music keeping me awake',
    'broken streetlight on my corner',
    'dangerous pothole on the road',
]
probe_embs = encoder.encode(probe_phrases, normalize_embeddings=True)

for i, phrase in enumerate(probe_phrases):
    sims = embeddings @ probe_embs[i]  # cosine since both normalized
    top_idx = np.argsort(-sims)[:5]
    print(f'\nprobe: "{phrase}"')
    for idx in top_idx:
        sim = sims[idx]
        text = df_sample.iloc[idx]['problem_detail']
        cat = df_sample.iloc[idx]['label_canonical']
        print(f'  {sim:.3f}  [{cat}]  {text}')

## Cell 5 - K-Means on BERT embeddings (sklearn, k=30 to match Phase 5)

We use sklearn rather than Spark MLlib here - 100K rows in numpy is way faster on driver than spinning up Spark for the same task. K=30 to match Phase 5's choice for direct silhouette comparison.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# sweep the same k values as phase 5 for direct comparison
ks = [5, 10, 15, 20, 25, 30]
results = []

for k in ks:
    t0 = time.time()
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=20)
    labels = km.fit_predict(embeddings)
    t_fit = time.time() - t0

    # silhouette on a 20K sample for speed (silhouette is O(n^2))
    sample_idx = np.random.RandomState(42).choice(len(embeddings), size=min(20_000, len(embeddings)), replace=False)
    score = silhouette_score(embeddings[sample_idx], labels[sample_idx])
    results.append((k, score, t_fit, km, labels))
    print(f'  k={k:>3}  silhouette={score:.4f}  fit_time={t_fit:.1f}s')

best_k, best_score, _, best_km, best_labels = max(results, key=lambda r: r[1])
print(f'\nbest k = {best_k} (silhouette = {best_score:.4f})')

## Cell 6 - BERT vs Word2Vec silhouette comparison plot

In [ ]:
import matplotlib.pyplot as plt
import json

# load phase 5's silhouette curve from the cluster_summary.json we saved there
phase5_path = Path('/content/project/dashboard/assets/cluster_summary.json')
if phase5_path.exists():
    p5_summary = json.loads(phase5_path.read_text())
    p5_sweep = {item['k']: item['silhouette'] for item in p5_summary['sweep']}
else:
    p5_sweep = {}
    print('warning: phase 5 summary not found - plot will only show bert curve')

bert_sweep = {r[0]: r[1] for r in results}

fig, ax = plt.subplots(figsize=(9, 5))
if p5_sweep:
    ax.plot(
        list(p5_sweep.keys()), list(p5_sweep.values()),
        marker='o', linewidth=2, color='#888888', label='Word2Vec (Phase 5)',
    )
ax.plot(
    list(bert_sweep.keys()), list(bert_sweep.values()),
    marker='s', linewidth=2.5, color='#57068C', label='BERT MiniLM (Phase 10)',
)
ax.set_xlabel('k')
ax.set_ylabel('silhouette score')
ax.set_title('K-Means silhouette: Word2Vec vs BERT MiniLM')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('/content/project/dashboard/assets/bert_vs_w2v.png', dpi=120)
plt.show()
print('saved bert_vs_w2v.png')

## Cell 7 - Top categories per BERT cluster

What complaint categories dominate each BERT cluster? Compare against Phase 5's Word2Vec cross-tab to see if BERT separates differently.

In [ ]:
df_sample['bert_cluster'] = best_labels

bert_cluster_categories = {}
for cid in range(best_k):
    sub = df_sample[df_sample['bert_cluster'] == cid]
    if len(sub) == 0:
        continue
    top = sub['label_canonical'].value_counts().head(3)
    total = top.sum()
    bert_cluster_categories[int(cid)] = [
        (name, int(cnt), round(100 * cnt / total, 1))
        for name, cnt in top.items()
    ]

print(f'top 3 official categories per BERT cluster (with % of cluster):')
for cid, cats in bert_cluster_categories.items():
    formatted = '  |  '.join(f'{name} ({pct}%)' for name, _cnt, pct in cats)
    print(f'  cluster {cid:>2}: {formatted}')

## Cell 8 - Save artifacts

We save the BERT embeddings as a parquet + the cluster assignments as JSON. The dashboard uses these for the BERT toggle in the Cluster Atlas tab.

In [ ]:
import datetime

# 1. embeddings + cluster labels - parquet on drive (too big for git)
out_path = '/content/drive/MyDrive/cs6513/bert_embeddings.parquet'
out_df = df_sample[['unique_key', 'label_canonical', 'problem_detail', 'bert_cluster']].copy()
out_df['embedding'] = list(embeddings)
out_df.to_parquet(out_path)
print(f'saved bert_embeddings.parquet to drive ({len(out_df):,} rows)')

# 2. lightweight cluster summary for the dashboard (no embeddings, just labels + counts)
summary = {
    'phase': 10,
    'computed_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'model': model_name,
    'n_encoded': int(len(df_sample)),
    'embedding_dim': int(embeddings.shape[1]),
    'encoding_time_sec': float(t_encode),
    'kmeans_sweep': [{'k': int(r[0]), 'silhouette': float(r[1])} for r in results],
    'best_k': int(best_k),
    'best_silhouette': float(best_score),
    'word2vec_silhouette_at_same_k': float(p5_sweep.get(best_k, 0)) if p5_sweep else None,
    'bert_cluster_categories': {str(k): v for k, v in bert_cluster_categories.items()},
}
with open('/content/project/dashboard/assets/bert_cluster_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('saved bert_cluster_summary.json')

# print the comparison
if p5_sweep:
    p5_at_k = p5_sweep.get(best_k, p5_sweep[max(p5_sweep.keys())])
    print(f'\nBERT silhouette at k={best_k}:     {best_score:.4f}')
    print(f'Word2Vec silhouette at k={best_k}: {p5_at_k:.4f}')
    print(f'BERT advantage: {(best_score - p5_at_k):+.4f}')

## Phase 10 - Done when

- Cell 1 confirms GPU is available.
- Cell 3 reports >500 docs/sec on T4 GPU.
- Cell 4 probe phrases return semantically related complaints (e.g., probe `rat infestation in the building` returns rodent complaints across multiple categories).
- Cell 5 sweep produces silhouette scores for direct comparison to Phase 5.
- Cell 6 saves the comparison plot.
- Cell 8 saves embeddings to Drive + summary JSON.

Save the print as `PRINT 8.pdf` (or whatever number you're on) and drop in the project directory. Phase 10 is the project's modernization story for the Q&A.